# Holistic convex optimization: algorithms and duality

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/holistic-convex-optimization.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/holistic-convex-optimization.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

The complete progression from a one-variable convex function to Lagrangian relaxations, with reusable reporting/projection strategies and actual Ipopt subproblem solves.


# Lagrange duality: subgradients, Newton steps and reusable strategies

Dependencies deliberately enter when first needed. Symbolic differentiation comes first, then plotting and progress tables, and finally Pyomo/Ipopt for solved Lagrangian subproblems. Reporting and projection strategies remain visible because this notebook teaches their implementation.

We keep all four original mathematical examples, including the alternative duals and abstract-model variants. The update direction below is explicit: minimize a convex function or maximize a concave Lagrange dual.


In [ ]:
def formula(x,exp,log):
    return exp(x)-log(x)


In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'sympy': 'sympy'}
ensure_packages(required_packages)


In [ ]:
import sympy as sp


In [ ]:
x=sp.Symbol('x',positive=True)
f=formula(x,sp.exp,sp.log)
fprime=sp.diff(f,x)
fprimeprime=sp.diff(fprime,x)


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'numpy': 'numpy'}
ensure_packages(required_packages)


In [ ]:
F=sp.lambdify(x,f,'numpy')
Fprime=sp.lambdify(x,fprime,'numpy')
Fprimeprime=sp.lambdify(x,fprimeprime,'numpy')


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'matplotlib': 'matplotlib'}
ensure_packages(required_packages)


In [ ]:
def Plot(functions,x,start,stop,descriptions=None):
    import numpy as np
    import matplotlib.pyplot as plt
    grid=np.linspace(start,stop,200)
    descriptions=descriptions or [str(i) for i in range(len(functions))]
    for function,description in zip(functions,descriptions):
        plt.plot(grid,function(grid),label='$'+description+'$',linewidth=2)
    plt.legend();plt.show()


This notebook intentionally tries to teach you some useful python as well. 
 * Main objective: reduce the amount of code.
 * Main strategy: avoid copy and paste.
 * Main technique: use strategies
 * Main pythonesque coding idiom: comprehensions (list, dictionary,…)
 * Main learning: abstraction is powerful in coding as well!


In [ ]:
Plot( [F,Fprime], x, .2, 2, [sp.latex(f) for f in [f,fprime] ] )

We now start defining and using strategies, which is a famous [design pattern]( https://en.wikipedia.org/wiki/Strategy_pattern).

Our first strategy is used to `show` the progress of our algorithms.

In [ ]:
def noshow( report, k, x, f, fprime ):
    return x

def ProjectOnInterval( x, minX=.01, maxX=100 ):
    return min( max( minX, x ), maxX )       

def Subgradient( x, f, fprime, minX, maxX, maxIter, show = noshow ):

    report = show( None, 0, x, f(x), fprime(x) )
    for k in range(1,maxIter):
        x = ProjectOnInterval( x + 1/k * ( -fprime(x) ), minX=minX, maxX=maxX )       
        report = show( report, k, x, f(x), fprime(x) )
    return report

In [ ]:
%time Subgradient(1,F,Fprime,.01,100,100)

The slide deck says that the first $113$ digits of our number are $0.56714329040978387299996866221035554975381578718651250813513107922304579308668456669321944696175229455763802497286$

# The power of strategies

We may **add** new strategies without touching the previous code!

In [ ]:
def trace( report, k, x, f, fprime ):
    print( '{:5d} {:23.8f} {:33.8f} {:33.8f}'.format( k, x, f, fprime ) )
    return x

In [ ]:
Subgradient(1,F,Fprime,.01,100,100,show=trace)

In [ ]:
from teaching_utils import ensure_packages
required_packages = {'pandas': 'pandas'}
ensure_packages(required_packages)


In [ ]:
def dataframe(report,k,x,f,fprime):
    import copy
    import pandas as pd
    row={'x':copy.deepcopy(x),'f':float(f),'fprime':copy.deepcopy(fprime)}
    if report is None:
        return pd.DataFrame([row],index=[k])
    report.loc[k]=row
    return report


In [ ]:
result = Subgradient(1,F,Fprime,.01,100,1000,dataframe)
result

In [ ]:
result[result.index>100].f.diff().plot()

In [ ]:
result[result.index>30].x.plot()

In [ ]:
result[abs(result.f.diff()) > 1e-20]

In [ ]:
Plot( [F,Fprime,Fprimeprime], x, .2, 2, [sp.latex(f) for f in [f,fprime,fprimeprime] ] )

Old strategies can be reused on new methods as well!

In [ ]:
def NewtonRaphson(x,f,fprime,fprimeprime,epsilon,maxIter,show=noshow):
    report=show(None,0,x,f(x),fprime(x))
    for k in range(1,maxIter):
        delta=fprime(x)/fprimeprime(x)
        # Preserve the positive domain of log(x); damp only if necessary.
        scale=1.0
        while x-scale*delta<=0:
            scale*=0.5
        x=x-scale*delta
        report=show(report,k,x,f(x),fprime(x))
        if abs(scale*delta)<epsilon:
            break
    return report


In [ ]:
%time NewtonRaphson(1,F,Fprime,Fprimeprime,1e-10,1000)

The stationary point solves x*exp(x)=1: the omega constant, approximately 0.5671432904097839. A small change in iterates is a stopping rule, not a proof of optimality. Check the derivative and the known solution as well; damping preserves the logarithm's domain.


In [ ]:
%time NewtonRaphson(10,F,Fprime,Fprimeprime,1e-15,1000,trace)

Powerful implementation of interior point methods, as `ipopt` for instance, take Newton steps along directions of descent. 

# When the convex optimization problem is a Lagrange Dual

## New strategies

In [ ]:
def freeProjection( u ):
    return u

The direction must match the objective. Use `sense='min'` for a convex minimization function and `sense='max'` for a concave Lagrange dual of a minimization primal. A relaxation returns its actual value and the appropriate (super)gradient; projection imposes the multiplier domain. For an equality multiplier the domain is free; for g(x)<=0 in a minimization primal it is nonnegative.


In [ ]:
def SubgradientForLagrangeDuals(u,relaxation,maxIter,epsilon=1e-10,project=freeProjection,show=noshow,sense='min'):
    import numpy as np
    if sense not in ('min','max'):
        raise ValueError('sense must be min or max')
    direction=-1 if sense=='min' else 1
    value,subgradient=relaxation(u)
    report=show(None,0,u,value,subgradient)
    for k in range(1,maxIter):
        newU=project(u+direction*subgradient/k)
        if np.linalg.norm(newU-u)<epsilon:
            break
        u=newU
        value,subgradient=relaxation(u)
        report=show(report,k,u,value,subgradient)
    return report


Note that the function above also works for the simple case we started with: minimizing a convex, differentiable function.

In [ ]:
def simple(u,f=F,fprime=Fprime):
  return f(u),fprime(u)

In [ ]:
SubgradientForLagrangeDuals( 1, simple, 100, project=ProjectOnInterval, show = noshow )

In our first example of a Lagrangean dual (the second example of this notebook) the relaxation can be solved by full enumeration (aka brute force). The original problem is:

$$
\begin{array}{rrcrcl}
\min    & -2x_1 & + &  x_2         \\
s.t.    &   x_1 & + &  x_2 & = & 3 \\
        &   x & \in & X            \\
\end{array}
$$
with $X = \{ (0,0),(0,4),(4,4),(4,0),(1,2),(2,1) \}$.

The relaxation is:
$$
\begin{array}{rcl}
\ell(u) = & \min  & -2x_1 + x_2 + u(x_1 + x_2 - 3)  \\
          & s.t.  & x  \in  X
\end{array}
$$

It can be rewritten as:
$$
\begin{array}{rcl}
\ell(u) = -3u + & \min  & (u-2)x_1 + (1+u)x_2  \\
                & s.t.  & x  \in  X
\end{array}
$$



In [ ]:
def relaxationExampleTwo( u ):
    import numpy as np
    X = [ (0,0),(0,4),(4,4),(4,0),(1,2),(2,1) ]
    c = np.array( [u-2, 1+u] )
    V = [np.dot(c,x) for x in X]
    i = np.argmin( V )
    x = X[ i ]
    v = V[ i ]
    s = sum(x)-3
    return -3*u + v, s

In [ ]:
%time SubgradientForLagrangeDuals( 0, relaxationExampleTwo, 1000, sense="max" )   

In [ ]:
result = SubgradientForLagrangeDuals( 0, relaxationExampleTwo, 100, sense="max", show=dataframe )   

In [ ]:
result.f.plot()

The dual bound need not improve monotonically. In this finite nonconvex example the best dual value is -6, while the feasible primal points give optimum -3: a genuine duality gap. Convergence of the dual algorithm does not remove that gap.


In [ ]:
X=[(0,0),(0,4),(4,4),(4,0),(1,2),(2,1)]
primal=min(-2*a+b for a,b in X if a+b==3)
dual,gradient=relaxationExampleTwo(2.0)
assert primal==-3 and dual==-6
print('Primal optimum, dual bound, gap:',primal,dual,primal-dual)


# Now we will treat the relaxations as abstract mathematical optimization models

We illustrate the `pyomo` abstract model (please also revisit the ExploringLAP notebook!) and use `ipopt` without loss of generality. 

On the last week we will again pay attention to the solver choice.

Notice in the examples below how we make use of expressions to reuse them: both defining the objective function and the subgradient. This way we avoid copy and paste.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'pyomo': 'pyomo'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from teaching_utils import install_coin_solvers, solve_checked
install_coin_solvers()


In [ ]:
def nonNegativeProjection( u ):
    import numpy as np # numpy.maximum is vectorized, unlike max
    return np.maximum(u,0)

$$
\begin{array}{rl}
\min    & x^2 + 3           \\
s.t.    & (x-1)(x-3) \leq 0 
\end{array}
$$

The relaxation is:
$$
\ell(u) = \min x^2 + 3 + u(x-1)(x-3) 
$$


In [ ]:
def modelExampleThree():
    model     = pyo.AbstractModel("ExampleThree")    
    model.x   = pyo.Var( within = pyo.Reals, initialize=0 )
    model.u   = pyo.Param(within=pyo.NonNegativeReals,default=0,mutable=True)
    model.f   = pyo.Expression( rule = lambda m : m.x**2 + 3 )
    model.g   = pyo.Expression( rule = lambda m : (m.x-1)*(m.x-3) )
    model.obj = pyo.Objective( rule = lambda m : m.f + m.u * m.g, sense=pyo.minimize )
    return model    

def relaxationExampleThree( u, model=modelExampleThree() ):
    data   = { None: dict( u = {None : u} ) }
    rel    = model.create_instance(data)
    result = solve_checked(rel,'ipopt')
    if result.solver.termination_condition == 'optimal':
        return pyo.value( rel.obj ), pyo.value( rel.g )
    return None,None

In [ ]:
%time result = SubgradientForLagrangeDuals( 0, relaxationExampleThree, 100, sense="max", epsilon = 0, project=nonNegativeProjection, show=dataframe )   

In [ ]:
result

In [ ]:
result.f.plot()

Looking at the primal you see that the optimal solution is $x=1$ with value $4$. The dual also converges (maximizes) to $4$ as expected, since we have strong duality in this case.

# Last problem of this notebook
$$
\begin{array}{rrcrcl}
\min    & x_1^2 & + & x_2^2          \\
s.t.    &   x_1 & + & x_2 & \geq & 4 \\
        &   x_1 & , & x_2 & \geq & 0
\end{array}
$$

The optimal solution is clearly among the points satisfying $x_1+x_2=4$ the one closest to $(0,0)$ and hence is that $(2,2)$, giving the value $8$ to the optimal solution. 

Our fourth example corresponds to relaxing the first constrain and keeping $X = \{ x : x\geq 0 \}$. 

Note that in the example below the relaxation implements the constraints in the domain of the variables only.

In [ ]:
def modelExampleFour():
    model     = pyo.AbstractModel("ExampleFour")    
    model.n   = 2
    model.idx = pyo.RangeSet(model.n)
    model.x   = pyo.Var( model.idx, within = pyo.NonNegativeReals, initialize=0 )
    model.u   = pyo.Param(within=pyo.NonNegativeReals,default=0,mutable=True)
    model.f   = pyo.Expression( rule = lambda m : sum(m.x[i]**2 for i in m.idx) )
    model.g   = pyo.Expression( rule = lambda m : 4-sum(m.x[i] for i in m.idx) )
    model.obj = pyo.Objective( rule = lambda m : m.f + m.u * m.g, sense=pyo.minimize )
    return model    

def relaxationExampleFour( u, model=modelExampleFour() ):
    data   = { None: dict( u = {None : u} ) }
    rel    = model.create_instance(data)
    result = solve_checked(rel,'ipopt')
    # import numpy as np
    # print( result.solver.termination_condition, np.array( [ pyo.value( rel.f ), pyo.value( rel.g ), pyo.value( rel.u ) ] + list( pyo.value(rel.x[i]) for i in rel.idx ) ) )
    if result.solver.termination_condition == 'optimal':
        return pyo.value( rel.obj ), pyo.value( rel.g )
    return None,None

In [ ]:
SubgradientForLagrangeDuals( 0, relaxationExampleFour, 5, sense="max", project=nonNegativeProjection, show=dataframe ).f.plot()   

# Same problem, other dual

$$
\begin{array}{rrcrcl}
\min    & x_1^2 & + & x_2^2          \\
s.t.    &   x_1 & + & x_2 & \geq & 4 \\
        &   x_1 &   &     & \geq & 0 \\
        &       &   & x_2 & \geq & 0
\end{array}
$$

We now relax all constraints.

$$
\ell(u) = \min x_1^2 + x_2^2 + u_1(4-x_1-x_2) + u_2(-x_1) + u_3(-x_2)
$$

In [ ]:
def modelExampleFourVersionTwo():
    model      = pyo.AbstractModel("ExampleFourVersionTwo")    
    model.idxX = pyo.Set()
    model.idxG = pyo.Set()
    model.b    = pyo.Param( model.idxG, default=0 )
    model.A    = pyo.Param( model.idxG, model.idxX, default=0 )
    model.x    = pyo.Var( model.idxX, within = pyo.Reals, initialize=0 )
    model.u    = pyo.Param(model.idxG, within=pyo.Reals, mutable=True)
    model.f    = pyo.Expression( rule = lambda m : sum(m.x[i]**2 for i in m.idxX) )
    model.g    = pyo.Expression( model.idxG, rule = lambda m, j : m.b[j] - sum( m.A[j,i]*m.x[i] for i in m.idxX ) )
    model.obj  = pyo.Objective( rule = lambda m : m.f + sum( m.u[j]*m.g[j] for j in m.idxG ), sense=pyo.minimize )
    return model    

def relaxationExampleFourVersionTwo( u, model=modelExampleFourVersionTwo() ):
    import numpy as np
    data   = { None:  { 'idxX' : [1,2]
                      , 'idxG' : [1,2,3] 
                      , 'u'    : {j+1 : u for j,u in enumerate(u) } 
                      , 'b'    : {1: 4}
                      , 'A'    : { (1,1) : 1, (1,2) : 1
                                 , (2,1) : 1 
                                 ,            (3,2) : 1  
                                 }
                      }
             }
    rel    = model.create_instance(data)
    result = solve_checked(rel,'ipopt')
    if result.solver.termination_condition == 'optimal':
        return pyo.value( rel.obj ), np.array( [ pyo.value( rel.g[j] ) for j in rel.idxG ] )
    return None,None

In [ ]:
def othertrace( report, k, x, f, fprime ):
    print( '{:3d} {} {:23.15g} {}'.format( k, x, f, fprime ) )
    return x

In [ ]:
def nonPositiveProjection( u ):
    import numpy as np # numpy.maximum is vectorized, unlike max
    return np.minimum(u,0)

In [ ]:
import numpy as np
np.set_printoptions(formatter={'float': lambda x: "{0:13.5g}".format(x)})
import pandas as pd
pd.options.display.float_format = '{:,.10f}'.format
result = SubgradientForLagrangeDuals( np.zeros(3), relaxationExampleFourVersionTwo, 100, sense="max", epsilon=5e-10, project=nonNegativeProjection, show=dataframe )   

In [ ]:
result

In [ ]:
result.f.diff()

In [ ]:
[np.linalg.norm(v) for v in result.x.diff().iloc[1:]]


One final version: with a model that is only abstract to delay the definition of $u$, all other parameters are defined in it.

In [ ]:
def modelExampleFourVersionThree():
    model      = pyo.AbstractModel("ExampleFourVersionThree")    
    model.idxX = pyo.RangeSet(2)
    model.idxG = pyo.RangeSet(3)
    model.b    = [4,0,0]
    model.A    = [[1,1],[1,0],[0,1]]
    model.x    = pyo.Var( model.idxX, within = pyo.Reals, initialize=0 )
    model.u    = pyo.Param(model.idxG, within=pyo.Reals, mutable=True)
    model.f    = pyo.Expression( rule = lambda m : sum(m.x[i]**2 for i in m.idxX) )
    model.g    = pyo.Expression( model.idxG, rule = lambda m, j : m.b[j-1] - sum( m.A[j-1][i-1]*m.x[i] for i in m.idxX ) )
    model.obj  = pyo.Objective( rule = lambda m : m.f + sum( m.u[j]*m.g[j] for j in m.idxG ), sense=pyo.minimize )
    return model    

def relaxationExampleFourVersionThree( u, model=modelExampleFourVersionThree() ):
    import numpy as np
    data   = { None:  { 'u' : {j+1 : u for j,u in enumerate(u) } } }
    rel    = model.create_instance(data)
    result = solve_checked(rel,'ipopt')
    if result.solver.termination_condition == 'optimal':
        return pyo.value( rel.obj ), np.array( [ pyo.value( rel.g[j] ) for j in rel.idxG ] )
    return None,None

In [ ]:
result = SubgradientForLagrangeDuals( np.zeros(3), relaxationExampleFourVersionThree, 100, sense="max", epsilon=5e-10, project=nonNegativeProjection, show=dataframe )   

In [ ]:
result

In [ ]:
assert abs(NewtonRaphson(1,F,Fprime,Fprimeprime,1e-12,100)-float(sp.LambertW(1)))<1e-10
for relaxation,multiplier,expected in [
    (relaxationExampleThree,1.0,4.0),
    (relaxationExampleFour,4.0,8.0),
    (relaxationExampleFourVersionTwo,np.array([4.,0.,0.]),8.0),
    (relaxationExampleFourVersionThree,np.array([4.,0.,0.]),8.0)]:
    value,gradient=relaxation(multiplier)
    assert abs(value-expected)<1e-5,(value,expected)
# The displayed iterations should approach the same bound without exceeding it.
assert result.f.max()<=8+1e-5 and result.f.max()>7.99
